# Load and process S. aureus data

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO, Phylo
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from fasta_to_bool import fasta_to_bool
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim

## Load and check `.fasta` data

In [ ]:
saureus_path = "../data/staph/saureus.fa"
saureus_count = 0
saureus_names = []

for record in SeqIO.parse(saureus_path, "fasta"):
    print(record.id, len(record.seq))
    saureus_names.append(record.id)
    saureus_count += 1

print(f"Total number of S. aureus sequences: {saureus_count}")

In [ ]:
core_path = "../data/staph/core_CDS_all.fa"
core_count = 0
core_id = []
core_lengths = []

for record in SeqIO.parse(core_path, "fasta"):
    print(record.id, len(record.seq))
    core_count += 1
    core_id.append(record.id)
    core_lengths.append(len(record.seq))

print(f"Total number of core sequences: {core_count}")

In [ ]:
core_lengths = np.array(core_lengths)
np.max(core_lengths), np.min(core_lengths), np.mean(core_lengths)

In [ ]:
ref_path = Path("../data/staph/references")
ref_count = 0
ref_lengths = []

for fasta_file in ref_path.glob("*.fa"):
    print(f"--- Processing File: {fasta_file.name} ---")

    for record in SeqIO.parse(fasta_file, "fasta"):
        print(record.id, len(record.seq))
        ref_count += 1
        ref_lengths.append(len(record.seq))

print(f"Total number of reference sequences: {ref_count}")

In [ ]:
ref_lengths = np.array(ref_lengths)
np.max(ref_lengths), np.min(ref_lengths), np.mean(ref_lengths)

## Blast info table

In [ ]:
result_df = pd.read_csv('../data/staph/blast_results.txt', sep='\t', header=None)
result_df

In [ ]:
result_df.columns = ['Query_ID', 'Subject_ID', 'Identity', 'Alignment_Length', 'Mismatches',
                     'Gap_Openings', 'Query_Start', 'Query_End', 'Subject_Start', 'Subject_End',
                     'E-value', 'Bit_Score']
result_df

## Trees from RAxML and ClonalFrameML

In [ ]:
start_tree = Phylo.read("../data/staph/RAxML_tree/saureus_start_tree.raxml.bestTree", "newick")
Phylo.draw_ascii(start_tree)

In [ ]:
result_tree = Phylo.read("../data/staph/CFML_tree/cfml_results.labelled_tree.newick", "newick")
Phylo.draw_ascii(result_tree)

In [ ]:
clonal_tree = Phylo.read("../data/staph/saureus_clonal.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

In [ ]:
# clonal_edge = np.loadtxt("../data/staph/clonal_edge.csv", delimiter=",", dtype=float)
# clonal_node_height = np.loadtxt("../data/staph/clonal_node_height.csv", delimiter=",", dtype=float)

In [ ]:
clonal_edge = np.loadtxt("../data/staph/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/staph/clonal_node_height.csv", delimiter=",", dtype=float)

## Load genome data and transfer to boolean matrix

In [ ]:
leaf_names = np.loadtxt("../data/staph/tip_names.csv", delimiter=",", dtype=str)
leaf_names = np.char.strip(leaf_names, '"')
len(leaf_names), len(saureus_names)

In [ ]:
bool_mat = fasta_to_bool("../data/staph/saureus.fa")
bool_mat.shape, bool_mat.dtype

In [ ]:
index_map = {string_id: idx for idx, string_id in enumerate(saureus_names)}
new_indices = [index_map[string_id] for string_id in leaf_names]
genomes_bool = bool_mat[new_indices]
genomes_bool.shape, genomes_bool.dtype

In [ ]:
# np.savetxt("../data/staph/genomes_bool.csv", genomes_bool, delimiter=",", fmt="%d")
# bool_mat = np.loadtxt("../data/staph/genomes_bool.csv", delimiter=",", dtype=bool)

## Compute estimated theta

In [ ]:
has_true = bool_mat.any(axis=0)
has_false = ~bool_mat.all(axis=0)
idx_seg = np.where(has_true & has_false)[0]

In [ ]:
count_S = idx_seg.size
count_S

In [ ]:
bool_mat.shape[1]

In [ ]:
np.sum(clonal_edge[:, 2])

theta / 2 = S / L / tree length

In [ ]:
float(count_S / bool_mat.shape[1] / np.sum(clonal_edge[:, 2]) * 2)

In [ ]:
1 / 0.00220427

In [ ]:
0.343035 * 0.001133205859498758

## Get clear dataframe for core gene positions

In [ ]:
core_id

In [ ]:
core_gene_df = pd.DataFrame({'Gene_ID': core_id, 'Gene_Length': core_lengths})
core_gene_df['Start_pos'] = None
core_gene_df['End_pos'] = None
core_gene_df['Alignment'] = True
core_gene_df

In [ ]:
clean_df = result_df[result_df['Subject_ID'].isin(leaf_names)]

In [ ]:
for i, gene_id in enumerate(core_id):
    df_filtered = clean_df[clean_df['Query_ID'] == gene_id].copy()
    df_sorted = df_filtered.sort_values(by='Bit_Score', ascending=False)
    df_best = df_sorted.drop_duplicates(subset='Subject_ID', keep='first')

    sstart_list = []
    send_list = []
    for subj_id in leaf_names:
        match_row = df_best[df_best['Subject_ID'] == subj_id]
            
        # Get coordinates
        sstart = int(match_row['Subject_Start'].iloc[0])
        send = int(match_row['Subject_End'].iloc[0])
        sstart_list.append(sstart)
        send_list.append(send)

        if sstart >= send:
            core_gene_df.loc[i, 'Alignment'] = False
    
    core_gene_df.loc[i, 'Start_pos'] = np.min(sstart_list)
    core_gene_df.loc[i, 'End_pos'] = np.max(send_list)

    if np.max(send_list) - np.min(sstart_list) + 1 != core_lengths[i]:
        print(f"Warning: Gene {gene_id} has inconsistent alignment lengths across subjects.")
    
    if i % 10 == 0:
        print(f"Processed {i+1}/{len(core_id)} genes.")

In [ ]:
core_gene_df['Alignment'].value_counts()

In [ ]:
# core_gene_df.to_csv("../data/staph/core_gene_info.csv", index=False)
# pd.read_csv("../data/staph/core_gene_info.csv")

## Get summary statistics for core genes

In [ ]:
core_gene_summary = np.full((len(core_id), 46), np.nan)
core_gene_summary

In [ ]:
np.random.seed(100)
clonal_tree = ClonalTree(n=110)

clonal_tree.edge = clonal_edge
clonal_tree.node_height = clonal_node_height
clonal_tree.height = np.max(clonal_node_height)
clonal_tree.length = np.sum(clonal_edge[:, 2])

In [ ]:
for i in tqdm(range(len(core_id)), desc="Processing genes"):
    gene_id = core_id[i]
    start_pos = core_gene_df.loc[i, 'Start_pos']
    end_pos = core_gene_df.loc[i, 'End_pos']
    seg_matrix = genomes_bool[:, start_pos-1:end_pos]  # Adjust for 0-based indexing
    summary_stats = segment_summary_stats(clonal_tree, seg_matrix)
    core_gene_summary[i, :] = summary_stats

In [ ]:
core_gene_summary_df = pd.DataFrame(core_gene_summary, 
                                    index=core_id)
core_gene_summary_df

In [ ]:
core_gene_summary_df.to_csv("../data/staph/core_gene_summary_stats.csv", index=True)